# StepManAI — Generate a stepchart

Fill in the form, press the ▶ button, get a `.zip`, unzip it into your
`Songs\StepManAI\` folder, reload songs. That's it.

- **spotify_link**: paste a Spotify track link — title + artist fill in automatically.
  Leave empty to upload your own audio file instead (it will prompt you).
- **level**: DDR difficulty 1-19. Or tick **all_difficulties** for a full 5-chart song.
- First run in a session takes ~1 min extra for setup; Drive access popup is normal.

In [ ]:
#@title Generate (press ▶)
spotify_link = "" #@param {type:"string"}
title = "" #@param {type:"string"}
artist = "" #@param {type:"string"}
level = 12 #@param {type:"slider", min:1, max:19, step:1}
all_difficulties = False #@param {type:"boolean"}
temperature = 0.9 #@param {type:"slider", min:0.5, max:1.3, step:0.05}
seed = 0 #@param {type:"integer"}

import glob, importlib, os, shutil, subprocess, sys
# one-time session setup
if not os.path.exists('/content/StepManAI'):
    print('setting up (~1 min)...')
    subprocess.run(['git', 'clone', '-q', 'https://github.com/Mrman67/StepManAI.git', '/content/StepManAI'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'soundfile', 'spotdl'], check=True)
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')
CKPT = '/content/drive/MyDrive/StepManAI/checkpoints'
assert os.path.exists(CKPT + '/placement.pt'), 'No trained models found — run the training notebook first.'
os.environ['STEPMANAI_CKPT'] = CKPT

if spotify_link.strip():
    audio = spotify_link.strip()
else:
    from google.colab import files
    print('Spotify link empty — upload an audio file (.mp3/.ogg/.wav):')
    up = files.upload()
    assert up, 'no file uploaded'
    audio = '/content/' + list(up)[0]
    if not os.path.exists(audio):
        shutil.move(list(up)[0], audio)

shutil.rmtree('/content/output', ignore_errors=True)
lv = 'all' if all_difficulties else str(level)
cmd = [sys.executable, '/content/StepManAI/generate.py', audio, '-l', lv,
       '--out', '/content/output', '--temperature', str(temperature)]
if title.strip(): cmd += ['-t', title.strip()]
if artist.strip(): cmd += ['-a', artist.strip()]
if seed: cmd += ['--seed', str(seed)]
r = subprocess.run(cmd)
assert r.returncode == 0, 'generation failed (see output above)'

song = sorted(glob.glob('/content/output/*'), key=os.path.getmtime)[-1]
zpath = shutil.make_archive('/content/' + os.path.basename(song), 'zip',
                            root_dir='/content/output', base_dir=os.path.basename(song))
from google.colab import files
files.download(zpath)
print('\nUnzip into OutFox Songs\\StepManAI\\ and reload songs.')